# GIN and FiLM curvature training across depths

**Role:** Training.

Train ordinary GIN models, size-conditioned FiLM models, or both across depths, widths, seeds and organoid folds. Every model receives the complete observed fate panel, with optional marker exclusivity. A shared timepoint filter selects the cohort before splitting; optional train-only baseline residualization removes organoid-level curvature. The notebook displays validation results and saves models, exact prepared datasets, splits and fitted transformations under `training_results/gin_depth_training/`. Permutation/ring controls belong in graph-control training, marker combinations in lineage-removal training, and missing-fate training in the masking notebook.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.

## Configuration

Choose the shared data filters, full or exclusive fate encoding, GNN depths and `model_types` (`gin`, `film`, or both). FiLM requires `log_num_cells` in `global_features`; it also passes the configured globals to the head. Depth zero is a control without message passing or FiLM layers. `n_folds=1` creates a single holdout. Set an optional `tag` to identify the run; output names include its creation timestamp.

In [ ]:
import copy, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from src.data.io import load_graph_dataset_from_dir, select_graph_targets
from src.data.metadata import attach_metadata_to_graphs, load_aux_metadata_for_dir, load_marker_names_from_dir
from src.data.filters import (filter_graphs_by_blacklist, load_graph_blacklist_from_dir,
    filter_graphs_by_timepoints, filter_graphs_by_sphericity, filter_graphs_by_numeric_metadata)
from src.data.preprocessing import interpolate_target_outliers_from_neighbors
from src.data.marker_exclusivity import exclusive_markers, EXCLUSIVITY_RULES
from src.training.preparation import prepare_fold
from src.models.gnn import GINCurvature, SizeFiLMGINCurvature
from src.models.size_models import seed_all
from src.training.loop import TrainConfig, train
from src.training.losses import WeightedLossTerm, edge_loss_term
from src.inference.predict import predict_targets
from src.artifacts.bundle import save_bundle, graph_membership
from src.artifacts.provenance import workflow_fingerprint
from src.artifacts.paths import training_run_path
from src.analysis.metrics.evaluation import compare_baseline_mse

In [ ]:
# Execution switches
RUN_TRAINING = True  # Enable fitting and saving new model checkpoints.

SETTINGS = dict(
    # Run identity
    tag='scMultiplexpaper',  # Optional purpose label; letters, digits, underscores and hyphens.

    # Dataset and targets
    dataset='after_cleanup',  # Source dataset folder under training_data/.
    target_indices=[0],  # Target columns to predict; [0] selects Gaussian curvature.

    # Data filters
    timepoints=['day4', 'day4p5', 'day4p5-more'],  # Shared cohort timepoints for train/validation; None keeps all, or use a list of labels.
    blacklist=True,  # Exclude organoids listed in the dataset blacklist.
    sphericity_max=0.93,  # Maximum retained sphericity; None disables this filter where supported.
    complexity_min=None,  # Minimum retained complexity; None disables this filter.
    interpolate_outliers=True,  # Replace extreme targets using neighboring cells.
    outlier_quantiles=[.005, .995],  # Lower and upper quantiles defining target outliers.

    # Fate encoding
    exclusive_markers=False,  # Apply ordered rules so each cell has at most one fate.

    # Model grid
    model_types=['gin'],  # Choose gin, film, or both; every model uses the complete fate panel.
    depths=[0, 1, 2, 3, 4],  # GNN message-passing depths or control-model ring radii.
    hidden_dims=[128],  # Hidden widths to compare in the model grid.
    seeds=[42],  # Model initialization and training seeds to compare.

    # Data splits
    n_folds=5,  # n_folds=1: single holdout; larger values: organoid folds.
    val_fraction=.2,  # Fraction of organoids held out when n_folds=1.
    split_seed=42,  # Random seed for organoid train/validation membership.

    # Target residualization
    residualize=False,  # Subtract the always-trained baseline from targets; False keeps original targets.
    baseline_features=['log_num_cells', 'log_surface_area', 'log_volume', 'log_volume_over_area'],  # Global features used only by the baseline model.

    # Model parameters
    film_hidden_dim=16,  # Width of the log-count FiLM conditioning network.
    global_features=['log_num_cells', 'log_surface_area', 'log_volume', 'log_volume_over_area'],  # Global columns appended to the prediction head.
    dropout=.1,  # Probability of hidden-feature dropout during training.
    norm='batch',  # Hidden normalization: batch, layer, or none.
    residual=True,  # Enable residual connections within the GNN.

    # Training hyperparameters
    lr=3e-4,  # Optimizer learning rate.
    weight_decay=1e-4,  # AdamW weight-decay coefficient.
    batch_size=128,  # Number of graphs or subgraphs per batch.
    max_epochs=500,  # Maximum optimization epochs per fitted model.
    patience=30,  # Early-stopping patience, in epochs.
    num_workers=0,  # DataLoader worker processes; 0 loads in the main process.
    loss_name='gaussian',  # Prediction loss; gaussian fits a mean and uncertainty.

    # Training loss
    edge_loss_weight=.2,  # Weight of the auxiliary edge-consistency penalty.
    edge_loss_params=dict(
        weighted=False,  # Weight edge penalties by the target difference.
        alpha=2.,  # Exponent controlling edge-loss weights.
        normalize_by='graph_std',  # Scale used to normalize the edge penalty.
        clip_weight=4.,  # Maximum weight assigned to one edge penalty.
    ),  # Options for computing the auxiliary edge penalty.

    # Baseline training
    baseline_hidden_dim=64,  # Hidden width of the global-only baseline.
    baseline_max_epochs=2000,  # Maximum training epochs for the global baseline.
    baseline_patience=30,  # Baseline early-stopping patience, in epochs.

)  # Explicit run configuration, saved alongside the models.

# Training hyperparameters
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Use CUDA when available, otherwise CPU.

# Paths and outputs
RUN_DIR = training_run_path(ROOT, 'gin_depth_training', SETTINGS['tag'])  # New run under training_results/<notebook>/<tag>_<timestamp>.

## Load and filter the cohort

Attach metadata and retain requested organoids. Optional outlier interpolation follows the existing cohort-level preprocessing; its fitted information is saved. Marker exclusivity changes only fate channels and preserves unassigned cells and graph topology.
Each active organoid filter prints kept/total counts and the retained fraction for each dataset and timepoint. Counts refer to the cohort entering that step, after any preceding filters.


In [ ]:
data_dir = ROOT/'training_data'/SETTINGS['dataset']  # Dataset directory supplying graphs and metadata.
graphs = load_graph_dataset_from_dir(str(data_dir))
attach_metadata_to_graphs(graphs, load_aux_metadata_for_dir(str(data_dir)), exclude_keys=None)
graphs = filter_graphs_by_timepoints(graphs, SETTINGS['timepoints'], print_summary=True)
graphs = select_graph_targets(graphs, target_indices=SETTINGS['target_indices'], inplace=False)
marker_names = load_marker_names_from_dir(str(data_dir))
if not marker_names or SETTINGS['target_indices'] != [0]:
    raise ValueError('This workflow requires named fate channels and scalar Gaussian curvature.')
if SETTINGS['blacklist']:
    graphs = filter_graphs_by_blacklist(graphs, load_graph_blacklist_from_dir(data_dir), print_summary=True)
if SETTINGS['sphericity_max'] is not None:
    graphs = filter_graphs_by_sphericity(graphs, max_sphericity=SETTINGS['sphericity_max'], print_summary=True)
if SETTINGS['complexity_min'] is not None:
    graphs = filter_graphs_by_numeric_metadata(graphs, key='complexity', min_value=SETTINGS['complexity_min'], allow_missing=False, print_summary=True)
if SETTINGS['exclusive_markers']:
    for g in graphs:
        g.x = torch.as_tensor(exclusive_markers(g.x.cpu().numpy(), marker_names), dtype=g.x.dtype)
if SETTINGS['interpolate_outliers']:
    graphs, outlier_info = interpolate_target_outliers_from_neighbors(graphs, clip_quantiles=SETTINGS['outlier_quantiles'])
else:
    outlier_info = None
assert len(graphs) >= 2 and len({g.organoid_str for g in graphs}) == len(graphs)
cohort = pd.DataFrame([dict(organoid_str=g.organoid_str, n_cells=len(g.x),
                          timepoint=str(g.meta.get('timepoint', 'unknown'))) for g in graphs])
display(cohort.groupby('timepoint').agg(organoids=('organoid_str','size'), median_N=('n_cells','median')))


## Organism-level splits

Partition the cohort already selected by the shared `timepoints` setting. Training and validation draw from the same allowed timepoints; no organoid can enter both partitions. Display fold membership counts by timepoint before fitting.

In [ ]:
rng = np.random.default_rng(SETTINGS['split_seed'])
order = rng.permutation(len(graphs))
fold_count = SETTINGS['n_folds']
if not 1 <= fold_count <= len(graphs):
    raise ValueError('Invalid number of folds')
if fold_count == 1:
    if not 0 < SETTINGS['val_fraction'] < 1:
        raise ValueError('Invalid validation fraction')
    val_sets = [order[:max(1, min(len(order)-1, round(len(order)*SETTINGS['val_fraction'])))]]
else:
    val_sets = np.array_split(order, fold_count)
splits = []
for fold, val in enumerate(val_sets):
    split = dict(fold=fold, train_indices=np.setdiff1d(order, val).tolist(),
                 val_indices=val.tolist())
    if not split['train_indices'] or not split['val_indices']:
        raise ValueError(f'Empty cohort split: fold {fold}')
    splits.append(split)
membership = [dict(fold=s['fold'], **graph_membership(
    train=[graphs[i] for i in s['train_indices']], validation=[graphs[i] for i in s['val_indices']])) for s in splits]
display(pd.DataFrame([dict(fold=s['fold'], role=role, timepoint=cohort.iloc[i].timepoint)
    for s in splits for role in ('train', 'val') for i in s[role + '_indices']])
    .groupby(['fold', 'role', 'timepoint']).size().rename('organoids').to_frame())

## Fit, evaluate and save

Prepare each fold once, fitting global standardization, the baseline (always trained) and the target transform on training organoids. Train all requested depths and GIN/FiLM variants on those exact inputs. Matching layers use the same initial weights for a given seed. Save each completed checkpoint immediately, together with its history and references to the shared fold inputs. The main notebook has no subset, permutation or masking training path.

In [ ]:
if not RUN_TRAINING:
    raise RuntimeError('Review settings, then set RUN_TRAINING=True to fit and save a new run.')
if not SETTINGS['model_types'] or len(set(SETTINGS['model_types'])) != len(SETTINGS['model_types']) or not set(SETTINGS['model_types']) <= {'gin', 'film'}:
    raise ValueError('model_types must contain gin, film, or both exactly once')
if 'film' in SETTINGS['model_types'] and 'log_num_cells' not in SETTINGS['global_features']:
    raise ValueError('FiLM requires log_num_cells in global_features')
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR/'settings.json').write_text(json.dumps(SETTINGS, indent=2))
(RUN_DIR/'splits.json').write_text(json.dumps(membership, indent=2))
(RUN_DIR/'exclusivity_rules.json').write_text(json.dumps(EXCLUSIVITY_RULES, indent=2))
cohort.to_csv(RUN_DIR/'cohort.csv', index=False)
provenance = workflow_fingerprint(ROOT/'experiments/training/gin_depth_training.ipynb')
(RUN_DIR/'provenance.json').write_text(json.dumps({'code_sha256':provenance,'device':DEVICE}, indent=2))
save_bundle(RUN_DIR/'cohort', dict(raw_graphs={g.organoid_str:g for g in graphs},
    all_marker_names=marker_names, settings=SETTINGS, outlier_info=outlier_info),
    splits=membership, provenance={'code_sha256':provenance})
records, scores, histories = [], [], {}
baseline_rows = []
for split in splits:
    seed_all(SETTINGS['split_seed'] + split['fold'])
    prepared = prepare_fold(graphs, split, global_features=SETTINGS['global_features'],
        residualize=SETTINGS['residualize'], baseline_features=SETTINGS['baseline_features'],
        baseline_kwargs=dict(hidden_dim=SETTINGS['baseline_hidden_dim'], max_epochs=SETTINGS['baseline_max_epochs'],
            patience=SETTINGS['baseline_patience'], batch_size=SETTINGS['batch_size'], num_workers=SETTINGS['num_workers'],
            train_kwargs=dict(device=DEVICE)))
    baseline_rows.extend(dict(fold=split['fold'], **row) for row in prepared['baseline_validation_mse'])
    pd.DataFrame(baseline_rows).to_csv(RUN_DIR/'baseline_validation_mse.csv', index=False)
    groups = prepared['groups']
    input_bundle = f"inputs/fold_{split['fold']}_all_intact"
    save_bundle(RUN_DIR/input_bundle, dict(**prepared, marker_names=marker_names),
                splits=membership[split['fold']], provenance={'code_sha256':provenance})
    for depth in SETTINGS['depths']:
        for width in SETTINGS['hidden_dims']:
            for seed in SETTINGS['seeds']:
                kw = dict(n_markers=len(marker_names), hidden_dim=width, num_layers=depth,
                          dropout=SETTINGS['dropout'], residual=SETTINGS['residual'],
                          norm=SETTINGS['norm'], global_dim=len(SETTINGS['global_features']))
                seed_all(seed)
                common_model = GINCurvature(**kw)
                for family in SETTINGS['model_types']:
                    if family == 'gin':
                        model = copy.deepcopy(common_model)
                    else:
                        seed_all(seed)
                        model = SizeFiLMGINCurvature(**kw, film_hidden_dim=SETTINGS['film_hidden_dim'],
                            size_feature_index=SETTINGS['global_features'].index('log_num_cells'))
                        model.load_state_dict(common_model.state_dict(), strict=False)
                    seed_all(seed)
                    cfg = TrainConfig(**{k:SETTINGS[k] for k in
                        ['lr','weight_decay','batch_size','max_epochs','patience','num_workers','loss_name']},
                        device=DEVICE, aux_losses=[WeightedLossTerm(name='edge', fn=edge_loss_term,
                        weight=SETTINGS['edge_loss_weight'], params=SETTINGS['edge_loss_params'])])
                    model, metrics, history = train(model, groups['train'], groups['val'], cfg)
                    key = f"{family}_all_intact_d{depth}_h{width}_f{split['fold']}_s{seed}"
                    record = dict(key=key, name=family, subset='all', signal='intact', depth=depth,
                                  hidden_dim=width, fold=split['fold'], seed=seed,
                                  bundle=f'models/{key}', input_bundle=input_bundle)
                    y, mu, _ = predict_targets(groups['val'], model, target_transform=prepared['transform'], device=DEVICE)
                    start = 0
                    for g in groups['val']:
                        end = start + len(g.x)
                        scores.append(dict(**record, organoid_str=g.organoid_str, n_cells=len(g.x),
                            mse=float(np.mean((np.asarray(y)[start:end]-np.asarray(mu)[start:end])**2))))
                        start = end
                    save_bundle(RUN_DIR/record['bundle'], dict(model=model.cpu(), record=record, history=history, metrics=metrics),
                                splits=membership[split['fold']], provenance={'code_sha256':provenance})
                    records.append(record); histories[key] = history
                    pd.DataFrame(records).to_json(RUN_DIR/'models.json', orient='records', indent=2)
                    compare_baseline_mse(scores, baseline_rows).to_csv(RUN_DIR/'validation_mse.csv', index=False)
(RUN_DIR/'complete.json').write_text(json.dumps({'models':len(records)}))
print('Saved run:', RUN_DIR)

## Review performance and reload

Compare organoid-averaged MSE in physical curvature units and early-stopping histories in transformed units. Residual and reconstructed-curvature MSE agree because the same saved offset is added to prediction and target. The final cell demonstrates loading without another training call.

Compare every model to the same fold-specific global baseline in physical-curvature MSE, averaging organoids equally. Negative `mse_minus_baseline` indicates improvement. The baseline is fitted and saved even when `residualize=False`; that switch controls only whether its prediction is subtracted from GNN targets.

In [ ]:
baseline_df = pd.DataFrame(baseline_rows)
scores_df = compare_baseline_mse(scores, baseline_rows)
display(scores_df.groupby(['name','subset','signal','depth'])[['mse','baseline_mse','mse_minus_baseline']].mean())
display(baseline_df.groupby('fold').mse.agg(['mean','std','count']))
fig, axes = plt.subplots(1, 2, figsize=(12,4))
for label, frame in scores_df.groupby(['name','subset','signal','hidden_dim']):
    curve = frame.groupby('depth').mse.mean()
    axes[0].plot(curve.index, curve.values, 'o-', label=str(label))
for key, history in histories.items():
    axes[1].plot(history['val_mae'], alpha=.5, label=key)
axes[0].set(xlabel='GNN depth', ylabel='Mean organoid validation MSE (curvature²)')
axes[0].axhline(baseline_df.mse.mean(), color='black', linestyle='--', label='Global baseline')
axes[0].legend(fontsize=7)
axes[1].set(xlabel='Epoch', ylabel='Validation MAE (training target units)')
plt.tight_layout(); plt.show()
# Reopen independently, with no training or preprocessing fit:
from src.artifacts.runs import AnalysisRun
saved = AnalysisRun(RUN_DIR)
display(saved.records)
restored = saved.select(saved.records.iloc[0].key)
print('Restored validation organoids:', len(restored['groups']['val']))